In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# تعلم الآلة Singularity - التصنيف: دالة Qiskit من Multiverse Computing
> **Note:** * دوال Qiskit ميزة تجريبية متاحة فقط لمستخدمي خطة IBM Quantum&reg; Premium Plan وFlex Plan وOn-Prem (عبر IBM Quantum Platform API) Plan. هي في مرحلة إصدار معاينة وقابلة للتغيير.

## نظرة عامة
بفضل دالة "تعلم الآلة Singularity - التصنيف"، يمكنك حل مشكلات تعلم الآلة الواقعية على أجهزة الكم دون الحاجة إلى خبرة كمية. هذه الدالة التطبيقية، المستندة إلى طرق التجميع، هي مصنِّف هجيني. تستغل الأساليب الكلاسيكية مثل التعزيز والتكييس والتجميع لتدريب التجميع الأولي. ثم تُستخدم الخوارزميات الكمية مثل المحلل الذاتي الكمي التنويعي (VQE) وخوارزمية التحسين الكمية التقريبية (QAOA) لتعزيز تنوع التجميع المُدرَّب وقدراته على التعميم وتعقيده الكلي.

على عكس حلول تعلم الآلة الكمية الأخرى، هذه الدالة قادرة على التعامل مع مجموعات بيانات واسعة النطاق بملايين الأمثلة والميزات دون أن تتقيد بعدد Qubits في وحدة المعالجة الكمية المستهدفة. عدد Qubits يحدد فقط حجم التجميع الذي يمكن تدريبه. كما أنها مرنة للغاية، وتُستخدم لحل مشكلات التصنيف في مجالات واسعة، تشمل التمويل والرعاية الصحية والأمن السيبراني.
وهي تحقق باستمرار دقة عالية في المشكلات الكلاسيكية الصعبة التي تتضمن مجموعات بيانات عالية الأبعاد وصاخبة وغير متوازنة.
![كيف يعمل](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
وهي مصممة لـ:
1. المهندسين وعلماء البيانات في الشركات الساعين إلى تعزيز منتجاتهم التقنية بدمج تعلم الآلة الكمي في منتجاتهم وخدماتهم،
2. الباحثين في مختبرات البحث الكمي الذين يستكشفون تطبيقات تعلم الآلة الكمي ويتطلعون إلى الاستفادة من الحوسبة الكمية لمهام التصنيف، و
3. الطلاب والمعلمين في المؤسسات التعليمية في مقررات مثل تعلم الآلة، ممن يسعون إلى إبراز مزايا الحوسبة الكمية.

يوضح المثال التالي وظائفها المختلفة، بما فيها `create` و`list` و`fit` و`predict`، ويبرهن على استخدامها في مشكلة اصطناعية تتكون من نصفَي دائرة متداخلَين، وهي مشكلة صعبة بطبيعتها بسبب حدودها القراري غير الخطي.


## وصف الدالة
تتيح دالة Qiskit هذه للمستخدمين حل مشكلات التصنيف الثنائي باستخدام مصنِّف التجميع المعزَّز كمياً من Singularity. خلف الكواليس، تستخدم نهجاً هجينياً لتدريب مجموعة من المصنِّفات كلاسيكياً على مجموعة البيانات المُعلَّمة، ثم تُحسِّنها لتحقيق أقصى تنوع وتعميم باستخدام خوارزمية التحسين الكمية التقريبية (QAOA) على وحدات المعالجة الكمية IBM&reg;. من خلال واجهة سهلة الاستخدام، يمكن للمستخدمين ضبط المصنِّف وفق متطلباتهم، وتدريبه على مجموعة البيانات التي يختارونها، وتوظيفه لعمل تنبؤات على مجموعة بيانات لم يسبق رؤيتها.

لحل مشكلة تصنيف عامة:
1. قم بمعالجة مجموعة البيانات مسبقاً، وقسِّمها إلى مجموعات للتدريب والاختبار. يمكنك اختيارياً تقسيم مجموعة التدريب إلى مجموعتين للتدريب والتحقق. يمكن تحقيق ذلك باستخدام [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. إذا كانت مجموعة التدريب غير متوازنة، يمكنك إعادة أخذ عينات منها لتوازن الفئات باستخدام [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. قم برفع مجموعات التدريب والتحقق والاختبار بشكل منفصل إلى مساحة تخزين الدالة باستخدام أسلوب `file_upload` من الكتالوج، مع تمرير المسار المناسب في كل مرة.
4. هيِّئ المصنِّف الكمي باستخدام إجراء `create` الخاص بالدالة، الذي يقبل المعاملات الفائقة مثل عدد وأنواع المتعلمين، والتنظيم (قيمة lambda)، وخيارات التحسين بما فيها عدد الطبقات ونوع المُحسِّن الكلاسيكي والنظيرة الكمية الخلفية وغيرها.
5. دِرِّب المصنِّف الكمي على مجموعة التدريب باستخدام إجراء `fit` الخاص بالدالة، مع تمرير مجموعة التدريب المُعلَّمة، ومجموعة التحقق إن وجدت.
6. اصنع تنبؤات على مجموعة الاختبار غير المسبوقة باستخدام إجراء `predict` الخاص بالدالة.
## النهج القائم على الإجراءات
تستخدم الدالة نهجاً قائماً على الإجراءات. يمكنك التفكير فيها كبيئة افتراضية تستخدم فيها الإجراءات لتنفيذ المهام أو تغيير حالتها. حالياً، تقدم الإجراءات التالية: [list](#1-list)، و[create](#2-create)، و[delete](#3-delete)، و[fit](#4-fit)، و[predict](#5-predict)، و[fit_predict](#6-fit-predict)، و[create_fit_predict](#7-create-fit-predict). يوضح المثال التالي إجراء `create_fit_predict`.

In [ ]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [1, 0, 0, 1, 0]
Probabilities (first five results):  [[0.16849563539001172, 0.8315043646099888], [0.8726393386620336, 0.12736066133796647], [0.795344837290717, 0.20465516270928288], [0.36822585748882725, 0.6317741425111725], [0.6656662698604361, 0.3343337301395641]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 7.945035696029663}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 82.41029238700867}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 77.3459484577179}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 71.27004957199097}}


### 1. List

The `list` action retrieves all stored classifiers in `*.pkl.tar` format from the shared data directory. You can also access the contents of this directory by using the `catalog.files()` method. In general, the list action searches for files with the `*.pkl.tar` extension in the shared data directory and returns them in a list format.

#### Inputs

|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | The name of the action from among `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` and `delete`. | Yes |

#### Usage

In [ ]:
job = singularity.run(action="list")

### 1. List

يسترجع إجراء `list` جميع المصنِّفات المُخزَّنة بصيغة `*.pkl.tar` من مجلد البيانات المشترك. يمكنك أيضاً الوصول إلى محتويات هذا المجلد باستخدام أسلوب `catalog.files()`. بشكل عام، يبحث إجراء list عن ملفات بامتداد `*.pkl.tar` في مجلد البيانات المشترك ويعيدها بتنسيق قائمة.

#### المدخلات

|     الاسم    |    النوع     | الوصف |   مطلوب  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | اسم الإجراء من بين `create` و`list` و`fit` و`predict` و`fit_predict` و`create_fit_predict` و`delete`. | نعم |

#### الاستخدام

In [ ]:
job = singularity.run(
    action="create",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
)

### 2. Create
يُنشئ إجراء `create` مصنِّفاً من النوع `quantum_classifier` المحدد باستخدام المعاملات المقدَّمة، ويحفظه في مجلد البيانات المشترك.

> **Note:** تدعم الدالة حالياً فقط `QuantumEnhancedEnsembleClassifier`.
#### المدخلات
|     الاسم    |    النوع     | الوصف | مطلوب  | الافتراضي |
|-------------|-------------|-------------|-----------|---------|
| `action` | `str` | اسم الإجراء من بين `create` و`list` و`fit` و`predict` و`fit_predict` و`create_fit_predict` و`delete`. | نعم | - |
| `name` | `str` | اسم المصنِّف الكمي، مثل `spam_classifier`. | نعم | - |
| `instance` | `str` | نسخة IBM. | نعم | - |
| `backend_name` | `str` | مورد الحوسبة في IBM. الافتراضي هو `None`، مما يعني استخدام النظيرة الخلفية ذات أقل الوظائف المعلَّقة. | لا | `None` |
| `quantum_classifier` | `str` | نوع المصنِّف الكمي، أي `QuantumEnhancedEnsembleClassifier`. | لا | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | عدد المتعلمين في التجميع. | لا | `10` |
| `learners_types` | `list` | أنواع المتعلمين. من الأنواع المدعومة: `DecisionTreeClassifier` و`GaussianNB` و`KNeighborsClassifier` و`MLPClassifier` و`LogisticRegression`. يمكن الاطلاع على مزيد من التفاصيل المتعلقة بكل منها في [وثائق scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | لا | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | نسب كل نوع متعلم في التجميع. | لا | `[1.0]` |
| `learners_options` | `list` | خيارات كل نوع متعلم في التجميع. للحصول على قائمة كاملة بالخيارات المتوافقة مع نوع/أنواع المتعلم المختارة، راجع [وثائق scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | لا | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` أو `list` | نوع/أنواع التنظيم المستخدَمة: `onsite` أو `alpha`. يتحكم `onsite` في الحد الموضعي حيث تؤدي القيم الأعلى إلى تجميعات أكثر تناثراً. يتحكم `alpha` في التوازن بين حدَّي التفاعل والموضع حيث تؤدي القيم الأدنى إلى تجميعات أكثر تناثراً. إذا قُدِّمت قائمة، فسيُدرَّب نموذج لكل نوع ويُختار الأفضل أداءً. | لا | `onsite` |
| `regularization` | `str` أو `float` أو `list` | قيمة التنظيم. محدودة بين `0` و`+inf` إذا كان regularization_type هو `onsite`. محدودة بين `0` و`1` إذا كان regularization_type هو `alpha`. إذا ضُبطت على `auto`، يُستخدم التنظيم التلقائي - يُجاد معامل التنظيم الأمثل بالبحث الثنائي بالنسبة المطلوبة من المصنِّفات المحددة إلى المصنِّفات الكلية (`regularization_desired_ratio`) والحد الأعلى لمعامل التنظيم (`regularization_upper_bound`). إذا قُدِّمت قائمة، فسيُدرَّب نموذج لكل قيمة ويُختار الأفضل أداءً. | لا | `0.01` |
| `regularization_desired_ratio` | `float` أو `list` | النسبة/النسب المطلوبة من المصنِّفات المحددة إلى المصنِّفات الكلية للتنظيم التلقائي. إذا قُدِّمت قائمة، فسيُدرَّب نموذج لكل نسبة ويُختار الأفضل أداءً. | لا | `0.75` |
| `regularization_upper_bound` | `float` أو `list` | الحد/الحدود العليا لمعامل التنظيم عند استخدام التنظيم التلقائي. إذا قُدِّمت قائمة، فسيُدرَّب نموذج لكل حد أعلى ويُختار الأفضل أداءً. | لا | `200` |
| `weight_update_method` | `str` | طريقة تحديث أوزان العينات من بين `logarithmic` و`quadratic`. | لا | `logarithmic` |
| `sample_scaling` | `boolean` | هل ينبغي تطبيق مقياس العينة. | لا | `False` |
| `prediction_scaling` | `float` | معامل المقياس للتنبؤات. | لا | `None` |
| `optimizer_options` | `dictionary` | خيارات مُحسِّن QAOA. قائمة بالخيارات المتاحة مُقدَّمة لاحقاً في هذه الوثائق. | لا | ... |
| `voting` | `str` | استخدام التصويت بالأغلبية (`hard`) أو متوسط الاحتمالات (`soft`) لتجميع تنبؤات/احتمالات المتعلمين. | لا | `hard` |
| `prob_threshold` | `float` | حد الاحتمال الأمثل. | لا | `0.5` |
| `random_state` | `integer` | التحكم في العشوائية لإمكانية التكرار. | لا | `None` |


- بالإضافة إلى ذلك، تُدرَج `optimizer_options` على النحو التالي:

|     الاسم    |    النوع     | الوصف | مطلوب  | الافتراضي |
|-------------|-------------|-------------|-----------|---------|
| `num_solutions` | `integer` | عدد الحلول | لا | `1024` |
| `reps` | `integer` | عدد التكرارات | لا | `4` |
| `sparsify` | `float` | حد التناثر | لا | `0.001` |
| `theta` | `float` | القيمة الأولية لـ theta، وهو معامل تنويعي لـ QAOA | لا | `None` |
| `simulator` | `boolean` | هل تُستخدم المحاكاة أم وحدة معالجة كمية | لا | `False` |
| `classical_optimizer` | `str` | اسم المُحسِّن الكلاسيكي لـ QAOA. جميع المحلِّلات التي توفرها SciPy، كما هو مُدرَج [هنا](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize)، قابلة للاستخدام. ستحتاج إلى ضبط `classical_optimizer_options` وفقاً لذلك | لا | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | خيارات المُحسِّن الكلاسيكي. للحصول على قائمة كاملة بالخيارات المتاحة، راجع [وثائق SciPy](https://docs.scipy.org/doc/scipy/reference/) | لا | `{"maxiter": 60}` |
| `optimization_level` | `integer` | عمق دائرة QAOA | لا | `3` |
| `num_transpiler_runs` | `integer` | عدد تشغيلات Transpiler | لا | `30` |
| `pass_manager_options` | `dictionary` | خيارات لإنشاء مدير مرور مُعيَّن مسبقاً | لا | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | خيارات المُقدِّر. للحصول على قائمة كاملة بالخيارات المتاحة، راجع [وثائق عميل Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) | لا | `None` |
| `sampler_options` | `dictionary` | خيارات أخذ العينات. للحصول على قائمة كاملة بالخيارات المتاحة، راجع [وثائق عميل Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options) | لا | `None` |

- القيم الافتراضية لـ `estimator_options` هي:

|     الاسم    |    النوع     | القيمة  |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- القيم الافتراضية لـ `sampler_options` هي:

|     الاسم    |    النوع     | القيمة |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### الاستخدام

In [ ]:
job = singularity.run(
    action="delete",
    name="classifier_name",  # specify the name of the classifier to delete here
)

#### التحقق من الصحة
- `name`:
    - يجب أن يكون الاسم فريداً، وسلسلة نصية لا تتجاوز 64 حرفاً.
    - يمكن أن يحتوي فقط على أحرف أبجدية رقمية وشرطات سفلية.
    - يجب أن يبدأ بحرف ولا ينتهي بشرطة سفلية.
    - يجب ألا يوجد مصنِّف بنفس الاسم في مجلد البيانات المشترك.
### 3. Delete
يزيل إجراء `delete` مصنِّفاً من مجلد البيانات المشترك.

#### المدخلات
|     الاسم    |    النوع     | الوصف |   مطلوب  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | اسم الإجراء. يجب أن يكون `delete`. | نعم |
| `name`      | `str`    | اسم المصنِّف المراد حذفه. | نعم |

#### الاستخدام

In [ ]:
job = singularity.run(
    action="fit",
    name="classifier_name",  # specify the name of the classifier to train here
    X=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
)

#### التحقق من الصحة
- `name`:
    - يجب أن يكون الاسم فريداً، وسلسلة نصية لا تتجاوز 64 حرفاً.
    - يمكن أن يحتوي فقط على أحرف أبجدية رقمية وشرطات سفلية.
    - يجب أن يبدأ بحرف ولا ينتهي بشرطة سفلية.
    - يجب أن يوجد مصنِّف بنفس الاسم في مجلد البيانات المشترك.
### 4. Fit
يُدرِّب إجراء `fit` مصنِّفاً باستخدام بيانات التدريب المقدَّمة.

#### المدخلات
|     الاسم    |    النوع     | الوصف |   مطلوب  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | اسم الإجراء. يجب أن يكون `fit`. | نعم |
| `name`      | `str`    | اسم المصنِّف المراد تدريبه. | نعم |
| `X`         | `array` أو `list` أو `str` | بيانات التدريب. يمكن أن تكون مصفوفة NumPy أو قائمة أو سلسلة نصية تُشير إلى اسم ملف في مجلد البيانات المشترك. | نعم |
| `y`         | `array` أو `list` أو `str` | قيم الهدف للتدريب. يمكن أن تكون مصفوفة NumPy أو قائمة أو سلسلة نصية تُشير إلى اسم ملف في مجلد البيانات المشترك. | نعم |
| `fit_params`| `dictionary`| معاملات إضافية لتمريرها إلى أسلوب `fit` الخاص بالمصنِّف. | لا |

##### fit_params
|     الاسم    |    النوع     | الوصف |   مطلوب  |   الافتراضي   |
|-------------|-------------|-------------|-------------|-------------|
| `validation_data` | `tuple` | بيانات التحقق والتسميات. | لا | `None` |
| `pos_label` | `integer` أو `str` | تسمية الفئة المراد تعيينها إلى 1. | لا | `None` |
| `optimization_data` | `str` | مجموعة البيانات لتحسين التجميع عليها. يمكن أن تكون إحدى: `train` أو `validation` أو `both`. | لا | `train` |

#### الاستخدام

In [ ]:
job = singularity.run(
    action="predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X=X_test,  # or "X_test.npy" if you uploaded it to the shared data directory
    options={
        "out": "output.json",
    },
)

#### التحقق من الصحة
- `name`:
    - يجب أن يكون الاسم فريداً، وسلسلة نصية لا تتجاوز 64 حرفاً.
    - يمكن أن يحتوي فقط على أحرف أبجدية رقمية وشرطات سفلية.
    - يجب أن يبدأ بحرف ولا ينتهي بشرطة سفلية.
    - يجب أن يوجد مصنِّف بنفس الاسم في مجلد البيانات المشترك.
### 5. Predict
يُستخدم إجراء `predict` للحصول على التنبؤات الصارمة والمرنة (الاحتمالات).

#### المدخلات
|     الاسم    |    النوع     | الوصف |   مطلوب  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | اسم الإجراء. يجب أن يكون `predict`. | نعم |
| `name`      | `str`    | اسم المصنِّف المراد استخدامه. | نعم |
| `X`         | `array` أو `list` أو `str` | بيانات الاختبار. يمكن أن تكون مصفوفة NumPy أو قائمة أو سلسلة نصية تُشير إلى اسم ملف في مجلد البيانات المشترك. | نعم |
| `options["out"]` | `str` | اسم ملف JSON الناتج لحفظ التنبؤات في مجلد البيانات المشترك. إذا لم يُقدَّم، تُعاد التنبؤات في نتيجة الوظيفة. | لا |

#### الاستخدام

In [ ]:
job = singularity.run(
    action="fit_predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "out": "output.json",
    },
)

#### التحقق من الصحة
- `name`:
    - يجب أن يكون الاسم فريداً، وسلسلة نصية لا تتجاوز 64 حرفاً.
    - يمكن أن يحتوي فقط على أحرف أبجدية رقمية وشرطات سفلية.
    - يجب أن يبدأ بحرف ولا ينتهي بشرطة سفلية.
    - يجب أن يوجد مصنِّف بنفس الاسم في مجلد البيانات المشترك.
- `options["out"]`:
    - يجب أن يكون اسم الملف فريداً، وسلسلة نصية لا تتجاوز 64 حرفاً.
    - يمكن أن يحتوي فقط على أحرف أبجدية رقمية وشرطات سفلية.
    - يجب أن يبدأ بحرف ولا ينتهي بشرطة سفلية.
    - يجب أن يحمل الامتداد `.json`.
### 6. Fit-predict
يُدرِّب إجراء `fit_predict` مصنِّفاً باستخدام بيانات التدريب ثم يستخدمه للحصول على التنبؤات الصارمة والمرنة (الاحتمالات).

#### المدخلات
|     الاسم    |    النوع     | الوصف |   مطلوب  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | اسم الإجراء. يجب أن يكون `fit_predict`. | نعم |
| `name`      | `str`    | اسم المصنِّف المراد استخدامه. | نعم |
| `X_train`   | `array` أو `list` أو `str` | بيانات التدريب. يمكن أن تكون مصفوفة NumPy أو قائمة أو سلسلة نصية تُشير إلى اسم ملف في مجلد البيانات المشترك. | نعم |
| `y_train`   | `array` أو `list` أو `str` | قيم الهدف للتدريب. يمكن أن تكون مصفوفة NumPy أو قائمة أو سلسلة نصية تُشير إلى اسم ملف في مجلد البيانات المشترك. | نعم |
| `X_test`    | `array` أو `list` أو `str` | بيانات الاختبار. يمكن أن تكون مصفوفة NumPy أو قائمة أو سلسلة نصية تُشير إلى اسم ملف في مجلد البيانات المشترك. | نعم |
| `fit_params`| `dictionary`| معاملات إضافية لتمريرها إلى أسلوب `fit` الخاص بالمصنِّف. | لا |
| `options["out"]` | `str` | اسم ملف JSON الناتج لحفظ التنبؤات في مجلد البيانات المشترك. إذا لم يُقدَّم، تُعاد التنبؤات في نتيجة الوظيفة. | لا |

#### الاستخدام

In [ ]:
job = singularity.run(
    action="create_fit_predict",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "save": True,
        "out": "output.json",
    },
)

#### التحقق من الصحة
- `name`:
    - يجب أن يكون الاسم فريداً، وسلسلة نصية لا تتجاوز 64 حرفاً.
    - يمكن أن يحتوي فقط على أحرف أبجدية رقمية وشرطات سفلية.
    - يجب أن يبدأ بحرف ولا ينتهي بشرطة سفلية.
    - يجب أن يوجد مصنِّف بنفس الاسم في مجلد البيانات المشترك.

- `options["out"]`:
    - يجب أن يكون اسم الملف فريداً، وسلسلة نصية لا تتجاوز 64 حرفاً.
    - يمكن أن يحتوي فقط على أحرف أبجدية رقمية وشرطات سفلية.
    - يجب أن يبدأ بحرف ولا ينتهي بشرطة سفلية.
    - يجب أن يحمل الامتداد `.json`.
### 7. Create-fit-predict
ينشئ إجراء `create_fit_predict` مصنِّفاً، ويُدرِّبه باستخدام بيانات التدريب المقدَّمة، ثم يستخدمه للحصول على التنبؤات الصارمة والمرنة (الاحتمالات).

#### المدخلات
| الاسم | النوع | الوصف | مطلوب |
|------|------|-------------|-----------|
| `action` | `str` | اسم الإجراء من بين `create` و`list` و`fit` و`predict` و`fit_predict` و`create_fit_predict` و`delete`. | نعم |
| `name` | `str` | اسم المصنِّف المراد استخدامه. | نعم |
| `quantum_classifier` | `str` | نوع المصنِّف، أي `QuantumEnhancedEnsembleClassifier`. الافتراضي هو `QuantumEnhancedEnsembleClassifier`. | لا |
| `X_train` | `array` أو `list` أو `str` | بيانات التدريب. يمكن أن تكون مصفوفة NumPy أو قائمة أو سلسلة نصية تُشير إلى اسم ملف في مجلد البيانات المشترك. | نعم |
| `y_train` | `array` أو `list` أو `str` | قيم الهدف للتدريب. يمكن أن تكون مصفوفة NumPy أو قائمة أو سلسلة نصية تُشير إلى اسم ملف في مجلد البيانات المشترك. | نعم |
| `X_test` | `array` أو `list` أو `str` | بيانات الاختبار. يمكن أن تكون مصفوفة NumPy أو قائمة أو سلسلة نصية تُشير إلى اسم ملف في مجلد البيانات المشترك. | نعم |
| `fit_params` | `dictionary` | معاملات إضافية لتمريرها إلى أسلوب `fit` الخاص بالمصنِّف. | لا |
| `options["save"]` | `boolean` | هل يُحفظ المصنِّف المُدرَّب في مجلد البيانات المشترك. الافتراضي هو `True`. | لا |
| `options["out"]` | `str` | اسم ملف JSON الناتج لحفظ التنبؤات في مجلد البيانات المشترك. إذا لم يُقدَّم، تُعاد التنبؤات في نتيجة الوظيفة. | لا |

#### الاستخدام

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

#### التحقق من الصحة
- `name`:
    - إذا كان `options["save"]` مضبوطاً على `True`:
        - يجب أن يكون الاسم فريداً، وسلسلة نصية لا تتجاوز 64 حرفاً.
        - يمكن أن يحتوي فقط على أحرف أبجدية رقمية وشرطات سفلية.
        - يجب أن يبدأ بحرف ولا ينتهي بشرطة سفلية.
        - يجب ألا يوجد مصنِّف بنفس الاسم في مجلد البيانات المشترك.

- `options["out"]`:
    - يجب أن يكون اسم الملف فريداً، وسلسلة نصية لا تتجاوز 64 حرفاً.
    - يمكن أن يحتوي فقط على أحرف أبجدية رقمية وشرطات سفلية.
    - يجب أن يبدأ بحرف ولا ينتهي بشرطة سفلية.
    - يجب أن يحمل الامتداد `.json`.
---
## ابدأ الآن
سجّل الدخول باستخدام [مفتاح API الخاص بك على IBM Quantum Platform](http://quantum.cloud.ibm.com/)، ثم اختر دالة Qiskit على النحو التالي:

In [3]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[-0.99958218  0.02890441]
 [ 0.03285169  0.24578719]
 [ 1.13127903 -0.49134546]
 [ 1.86951286  0.00608971]
 [ 0.20190413  0.97940529]
 [ 0.8831311   0.46912627]
 [-0.10819442  0.99412975]
 [-0.20005727  0.97978421]
 [-0.78775705  0.61598607]
 [ 1.82453236 -0.0658148 ]]
Targets: [0 1 1 1 0 0 0 0 0 1]


## مثال
في هذا المثال، ستستخدم دالة "Singularity Machine Learning - Classification" لتصنيف مجموعة بيانات تتكون من نصفَي دائرة متشابكَين على شكل قمر. مجموعة البيانات هذه اصطناعية وثنائية الأبعاد، ومُصنَّفة بتسميات ثنائية. صُمِّمت لتكون تحديًا حقيقيًا لخوارزميات مثل التجميع القائم على النقاط المرجعية والتصنيف الخطي.
![مجموعة بيانات القمر](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
من خلال هذه العملية، ستتعلم كيفية إنشاء المصنِّف وضبطه على بيانات التدريب، ثم استخدامه للتنبؤ على بيانات الاختبار، وأخيرًا حذفه عند الانتهاء.
قبل البدء، تحتاج إلى تثبيت [scikit-learn](https://scikit-learn.org/). ثبِّته باستخدام الأمر التالي:

In [4]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


اتبع الخطوات التالية:

1) أنشئ مجموعة البيانات الاصطناعية باستخدام دالة [make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) من [scikit-learn](https://scikit-learn.org/).
2) ارفع مجموعة البيانات الاصطناعية التي أنشأتها إلى [دليل البيانات المشتركة](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html).
3) أنشئ المصنِّف المُعزَّز كميًا باستخدام إجراء [create](#2-create).
4) اعرض قائمة بمصنِّفاتك باستخدام إجراء [list](#1-list).
5) درِّب المصنِّف على بيانات التدريب باستخدام إجراء [fit](#4-fit).
6) استخدم المصنِّف المدرَّب للتنبؤ على بيانات الاختبار باستخدام إجراء [predict](#5-predict).
7) احذف المصنِّف باستخدام إجراء [delete](#3-delete).
8) نظِّف البيانات بعد الانتهاء.

**الخطوة 1.** استورد الوحدات الضرورية وأنشئ مجموعة البيانات الاصطناعية، ثم قسِّمها إلى مجموعتَي تدريب واختبار.

In [5]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [6]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}
['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar', 'my_classifier.pkl.tar']


**الخطوة 2.** احفظ مجموعتَي بيانات التدريب والاختبار المُصنَّفتَين على قرصك المحلي، ثم ارفعهما إلى [دليل البيانات المشتركة](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html).

In [7]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 8.45469617843628}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 69.4949426651001}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 73.01881957054138}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 75.4787163734436}}}}


**Step 5.** Obtain predictions and probabilities from the quantum-enhanced classifier using the [predict](#5-predict) action.

In [8]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 1, 0, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 0.0], [0.0, 1.0]]


**الخطوة 3.** أنشئ مصنِّفًا مُعزَّزًا كميًا باستخدام إجراء [create](#2-create).

In [9]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


**Step 7.** Clean up local and shared data directories.

In [ ]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

## Benchmarks

These benchmarks show that the classifier can achieve extremely high accuracies on challenging problems. They also show that increasing the number of learners in the ensemble (number of qubits) can lead to increased accuracy.

"Classical accuracy" refers to the accuracy obtained using corresponding classical state of the art which, in this case, is an AdaBoost classifier based on an ensemble of size 75. "Quantum accuracy", on the other hand, refers to the accuracy obtained using the "Singularity Machine Learning - Classification".

| Problem | Dataset Size | Ensemble Size | Number of Qubits | Classical Accuracy | Quantum Accuracy | Improvement |
|-------------|-------------|-------------|-------------|-------------|-------------|-------------|
| Grid stability | 5000 examples, 12 features | 55 | 55 |  76% | 91% | 15% |
| Grid stability | 5000 examples, 12 features | 65 | 65 |  76% | 92% | 16% |
| Grid stability | 5000 examples, 12 features | 75 | 75 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 85 | 85 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 100 | 100 |  76% | 95% | 19% |

----

As quantum hardware evolves and scales, the implications for our quantum classifier become increasingly significant. While the number of qubits does impose limitations on the size of the ensemble that can be utilized, it does not restrict the volume of data that can be processed. This powerful capability enables the classifier to efficiently handle datasets containing millions of data points and thousands of features. Importantly, the constraints related to ensemble size can be addressed through the implementation of a large-scale version of the classifier. By leveraging an iterative outer-loop approach, the ensemble can be dynamically expanded, enhancing flexibility and overall performance. However, it's worth noting that this feature has not yet been implemented in the current version of the classifier.

## Changelog

### 4 June 2025
- Upgraded `QuantumEnhancedEnsembleClassifier` with the following updates:
  - Added onsite/alpha regularization. You can specify `regularization_type` to be `onsite` or `alpha`
  - Added auto-regularization. You can set `regularization` to `auto` to use auto-regularization
  - Added `optimization_data` parameter to the `fit` method to choose optimization data for quantum optimization. You can use one of these options: `train`, `validation`, or `both`
  - Improved overall performance
- Added detailed status tracking for running jobs

### 20 May 2025
- Standardized error handling

### 18 March 2025
- Upgraded qiskit-serverless to 0.20.0 and base image to 0.20.1

### 14 February 2025
- Upgraded base image to 0.19.1

### 6 February 2025
- Upgraded qiskit-serverless to 0.19.0 and base image to 0.19.0

### 13 November 2024
- Release of Singularity Machine Learning - Classification

## Get support

For any questions, [reach out to Multiverse Computing](mailto:singularity@multiversecomputing.com).

Be sure to include the following information:

- The Qiskit Function Job ID (`job.job_id`)
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Multiverse Computing's Singularity Machine Learning Classification function](https://quantum.cloud.ibm.com/functions?id=multiverse-singularity).
- Review [Leclerc, L., et al. (2023). Financial risk management on a neutral atom quantum processor. Physical Review Research, 5, 043117.](https://journals.aps.org/prresearch/pdf/10.1103/PhysRevResearch.5.043117)

</Admonition>